<a href="https://colab.research.google.com/github/vaishnaviii7557/pet-classifier/blob/master/01_dataset_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))


2.21.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from torchvision.datasets import OxfordIIITPet
from collections import Counter

data = OxfordIIITPet(root="data", split="trainval",
                     target_types="binary-category", download=True)
labels = [data[i][1] for i in range(len(data))]
print(Counter(labels))

Counter({1: 2492, 0: 1188})


In [ ]:
import os
import pandas as pd

root = "data/oxford-iiit-pet"
print(os.listdir(root))

['annotations.tar.gz', 'images', 'images.tar.gz', 'annotations']


In [ ]:
rows = []
with open(f"{root}/annotations/list.txt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        name, class_id, species, breed_id = line.split()
        rows.append((name, int(species), int(breed_id)))

df = pd.DataFrame(rows, columns=["name", "species", "breed_id"])
df["label"] = df["species"].map({1: "cat", 2: "dog"})
print(len(df))
print(df["label"].value_counts())
df.head()

7349
label
dog    4978
cat    2371
Name: count, dtype: int64


,name,species,breed_id,label
0,Abyssinian_100,1,1,cat
1,Abyssinian_101,1,1,cat
2,Abyssinian_102,1,1,cat
3,Abyssinian_103,1,1,cat
4,Abyssinian_104,1,1,cat


In [ ]:
df["is_cat_by_name"] = df["name"].str[0].str.isupper()
print((df["is_cat_by_name"] == (df["label"] == "cat")).all())

True


In [ ]:
from PIL import Image

for n in df["name"].sample(10, random_state=0):
    im = Image.open(f"{root}/images/{n}.jpg")
    print(n, im.size, im.mode)

great_pyrenees_120 (500, 419) RGB
Egyptian_Mau_16 (241, 200) RGB
havanese_70 (500, 351) RGB
Abyssinian_136 (333, 500) RGB
wheaten_terrier_102 (500, 333) RGB
great_pyrenees_125 (333, 500) RGB
Egyptian_Mau_39 (310, 271) RGB
leonberger_130 (500, 333) RGB
newfoundland_143 (500, 375) RGB
British_Shorthair_92 (500, 335) RGB


In [ ]:
from PIL import Image

bad, modes, sizes = [], {}, []
for n in df["name"]:
    path = f"{root}/images/{n}.jpg"
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            modes[im.mode] = modes.get(im.mode, 0) + 1
            sizes.append(im.size)
    except Exception as e:
        bad.append((n, str(e)))

print("broken files:", len(bad))
print("color modes:", modes)

broken files: 0
color modes: {'RGB': 7346, 'RGBA': 3}


In [ ]:
w = [s[0] for s in sizes]
h = [s[1] for s in sizes]
print("width:", min(w), max(w))
print("height:", min(h), max(h))

width: 114 3264
height: 103 2606


In [ ]:
import hashlib

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df["hash"] = [file_hash(f"{root}/images/{n}.jpg") for n in df["name"]]
print("duplicate files:", df["hash"].duplicated().sum())

duplicate files: 0


In [ ]:
cats = df[df["label"] == "cat"]
dogs = df[df["label"] == "dog"].sample(len(cats), random_state=42)
bal = pd.concat([cats, dogs]).reset_index(drop=True)
print(bal["label"].value_counts())

label
cat    2371
dog    2371
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(bal, test_size=0.30,
                               stratify=bal["label"], random_state=42)
val, test = train_test_split(temp, test_size=0.50,
                             stratify=temp["label"], random_state=42)
print(len(train), len(val), len(test))

3319 711 712


In [ ]:
print(set(train["name"]) & set(val["name"]))
print(set(train["name"]) & set(test["name"]))
print(set(val["name"]) & set(test["name"]))


set()
set()
set()


In [ ]:
import os

for split_name, part in [("train", train), ("val", val), ("test", test)]:
    for _, r in part.iterrows():
        folder = f"dataset/{split_name}/{r['label']}"
        os.makedirs(folder, exist_ok=True)
        im = Image.open(f"{root}/images/{r['name']}.jpg").convert("RGB")
        im.save(f"{folder}/{r['name']}.jpg")

In [ ]:
for s in ["train", "val", "test"]:
    for c in ["cat", "dog"]:
        print(s, c, len(os.listdir(f"dataset/{s}/{c}")))

train cat 1659
train dog 1660
val cat 356
val dog 355
test cat 356
test dog 356


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!zip -rq dataset.zip dataset
!cp dataset.zip /content/drive/MyDrive/

train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)

Mounted at /content/drive
